# Mergen-SNN: Google Colab Kesintisiz Eğitim & Checkpoint-Resume Hattı

Bu notebook, **Mergen-SNN** (~20M parametreye kadar ölçeklenen Stage 4 Small SNN) dil modelinin Google Colab ortamında kesintisiz eğitilmesi için tasarlanmıştır.

### Özellikler:
- **Google Drive Entegrasyonu:** Checkpoint'ler doğrudan Google Drive üzerine atomik olarak kaydedilir.
- **Kopma Dayanıklılığı:** Colab bağlantısı kopsa veya oturum zaman aşımına uğrasa bile `--resume auto` ile en son `step*.pt` dosyasından devam eder.
- **Symlink Mimarisi:** Model kodunda hiçbir değişiklik gerektirmeden `checkpoints/` dizini Drive'a sembolik linklenir.

## 1. Google Drive Bağlantısı
Google Drive'ı bağlayarak eğitim checkpoint'lerinin kalıcı depolamada saklanmasını sağlayın.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Proje Kurulumu ve Bağımlılıklar
Depo klonlanır ve `uv` paket yöneticisi ile tüm Cython, PyTorch ve SNN bağımlılıkları yüklenir.

In [ ]:
!git clone https://github.com/haremir/mergen-snn.git /content/mergen-snn
%cd /content/mergen-snn
!pip install -q uv
!uv sync
# Cython uzantılarını derle
!uv run python src/mergen/cython_ext/setup.py build_ext --inplace

## 3. Checkpoint Dizinini Google Drive'a Bağla (Symlink)
Yerel `checkpoints/` klasörü yerine `/content/drive/MyDrive/mergen-snn/checkpoints` dizini sembolik linklenir.

In [ ]:
import os
import shutil

drive_ckpt = "/content/drive/MyDrive/mergen-snn/checkpoints"
local_ckpt = "/content/mergen-snn/checkpoints"

os.makedirs(drive_ckpt, exist_ok=True)

if os.path.islink(local_ckpt):
    os.unlink(local_ckpt)
elif os.path.exists(local_ckpt):
    for f in os.listdir(local_ckpt):
        src = os.path.join(local_ckpt, f)
        dst = os.path.join(drive_ckpt, f)
        if not os.path.exists(dst) and os.path.isfile(src):
            shutil.copy2(src, dst)
    shutil.rmtree(local_ckpt)

os.symlink(drive_ckpt, local_ckpt)
print(f"Symlink başarıyla oluşturuldu:")
print(f"  {local_ckpt} -> {drive_ckpt}")
print(f"Drive üzerindeki mevcut checkpoint sayısı: {len(os.listdir(drive_ckpt))}")

## 4. Eğitimi Başlat (Otomatik Resume ile)
`--resume auto` bayrağı ile eğitim Drive'daki en son checkpoint'ten devam eder.

In [ ]:
%cd /content/mergen-snn
!uv run python scripts/train_small.py --config configs/stage4_small.yaml --resume auto

## 5. Güvenli Kapatma ve Checkpoint Senkronizasyon Kontrolü
Oturum kapanmadan önce kaydedilen tüm checkpoint'leri listeler ve bütünlüğünü doğrular.

In [ ]:
import glob
import os

print("=== Google Drive Checkpoint Durumu ===")
ckpts = glob.glob("/content/drive/MyDrive/mergen-snn/checkpoints/*.pt")
if not ckpts:
    print("Henüz checkpoint kaydedilmemiş.")
else:
    for c in sorted(ckpts):
        sz_mb = os.path.getsize(c) / (1024 * 1024)
        print(f"  - {os.path.basename(c)}: {sz_mb:.2f} MB")
    print(f"\nToplam {len(ckpts)} checkpoint Google Drive üzerinde güvence altında!")